<a href="https://colab.research.google.com/github/ipeirotis/dealing_with_data/blob/master/01-Pandas/C-Association_Analysis_Lift_and_LogOdds.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# C1: Association Analysis — Lift and Log-Odds

## Finding Patterns in Data

A common analytical question is: **"Are two things associated?"**

Examples:
- Do people who buy diapers also buy beer? (Market basket analysis)
- Are certain books preferred by liberals vs. conservatives?
- Do users who click on ad A also tend to buy product B?

Raw counts can be misleading because groups have different sizes. This notebook introduces two powerful metrics for measuring association:

| Metric | Interpretation | Value Range |
|--------|----------------|-------------|
| **Lift** | How much more likely is X given Y? | 0 to ∞ (1 = no association) |
| **Log-Odds** | Logarithm of lift | -∞ to +∞ (0 = no association) |

## Learning Objectives

By completing this notebook, you will be able to:
1. Normalize raw counts to make fair comparisons
2. Calculate and interpret lift
3. Calculate and interpret log-odds
4. Use these metrics to discover hidden patterns in data
5. Build simple scoring models based on associations

## Real-World Applications

- **Recommendation systems**: "Customers who bought X also bought Y"
- **A/B testing**: Which user segments respond to which treatments?
- **Marketing**: Which products appeal to which demographics?
- **Social science**: What behaviors correlate with what attitudes?

---

## Setup

In [ ]:
# Install BigQuery library
!pip install -q google-cloud-bigquery

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%config InlineBackend.figure_format = 'retina'
plt.rcParams['figure.figsize'] = [10, 5]

# Authenticate with Google Cloud
from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()

# Set your project ID
project_id = 'ipeirotis-hrd'  # Replace with your project ID
bq = bigquery.Client(project=project_id)

print("✓ Setup complete!")

---

## Part 1: The Problem with Raw Counts

Let's explore the relationship between **gender** and **political views** using Facebook profile data.

### Load the Data

In [ ]:
query = '''
SELECT Sex, PoliticalViews, COUNT(*) AS cnt
FROM `nyu-datasets.facebook.Profiles`
WHERE Sex IS NOT NULL AND PoliticalViews IS NOT NULL
GROUP BY Sex, PoliticalViews
'''

df = bq.query(query).to_dataframe()
df

In [ ]:
# Create a pivot table
pivot = pd.pivot_table(
    data=df,
    index='PoliticalViews',
    columns='Sex',
    values='cnt',
    aggfunc='sum'
)

# Order political views on a spectrum
order = ['Very Liberal', 'Liberal', 'Moderate', 'Conservative',
         'Very Conservative', 'Libertarian', 'Apathetic', 'Other']
pivot = pivot.reindex(order)

pivot

In [ ]:
# Plot raw counts
pivot.plot(kind='bar', figsize=(10, 5))
plt.title('Political Views by Gender (Raw Counts)')
plt.ylabel('Number of Users')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

### The Problem

Looking at raw counts, it appears that **women appear to be more numerous**  across almost every category.

Let's check the total counts:

In [ ]:
print("Total users by gender:")
print(pivot.sum())
print(f"\nRatio: {pivot.sum()['Female'] / pivot.sum()['Male']:.2f}x more females")

**The dataset has ~30% more females than males!** Raw counts are misleading.

We need to **normalize** the data to make fair comparisons.

---

## Part 2: Normalization

### Column Normalization (Within Gender)

Divide each column by its sum so columns add up to 100%.

**Question answered**: "What percentage of males/females hold each political view?"

In [ ]:
# Normalize columns: each column sums to 1
normalized_by_gender = pivot / pivot.sum()
normalized_by_gender

In [ ]:
# Verify: columns sum to 1
normalized_by_gender.sum()

In [ ]:
# Plot normalized data
normalized_by_gender.plot(kind='bar', figsize=(10, 5))
plt.title('Political Views by Gender (Normalized)')
plt.ylabel('Proportion within Gender')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

**Insight**: After normalization, we see that males are more liberal while males are more conservative.

### Row Normalization (Within Political View)

**Question answered**: "What percentage of liberals/conservatives are male/female?"

In [ ]:
# Normalize rows: each row sums to 1
# Use .div() with axis='index' or transpose trick
normalized_by_politics = pivot.T / pivot.T.sum()
normalized_by_politics = normalized_by_politics.T

normalized_by_politics

In [ ]:
# Stacked bar chart shows composition
normalized_by_politics.plot(kind='barh', stacked=True, figsize=(10, 5),
                            color=['pink', 'lightblue'])
plt.title('Gender Composition by Political View')
plt.xlabel('Proportion')
plt.legend(title='Gender')
plt.tight_layout()

---

## Part 3: Lift — Measuring Association Strength

**Lift** measures how much more (or less) likely something is compared to the baseline.

$$\text{Lift}(A, B) = \frac{P(A | B)}{P(A)}$$

- **Lift = 1**: No association (A and B are independent)
- **Lift > 1**: Positive association (B increases likelihood of A)
- **Lift < 1**: Negative association (B decreases likelihood of A)

### Example: Books and Political Views

Let's find which books are preferred by liberals vs. conservatives.

In [ ]:
# Get book preferences by political view
books_query = '''
SELECT B.Book, P.PoliticalViews, COUNT(*) AS cnt
FROM `nyu-datasets.facebook.Profiles` P
JOIN `nyu-datasets.facebook.FavoriteBooks` B ON B.ProfileID = P.ProfileID
WHERE PoliticalViews IS NOT NULL
  AND B.Book IS NOT NULL
  AND PoliticalViews IN ('Liberal', 'Conservative')
  AND B.Book IN (
      SELECT Book
      FROM `nyu-datasets.facebook.FavoriteBooks` B
      JOIN `nyu-datasets.facebook.Profiles` P ON B.ProfileID = P.ProfileID
      WHERE P.PoliticalViews IN ('Liberal', 'Conservative')
      GROUP BY Book
      HAVING COUNT(DISTINCT P.ProfileID) > 10
  )
GROUP BY B.Book, P.PoliticalViews
'''

df_books = bq.query(books_query).to_dataframe()
print(f"Loaded {len(df_books)} book-politics combinations")
df_books.head(10)

In [ ]:
# Pivot: rows = books, columns = political views
books_pivot = df_books.pivot_table(
    index='Book',
    columns='PoliticalViews',
    values='cnt'
).fillna(0)  # Replace NaN with 0

books_pivot.head(10)

### Get Baseline Counts

In [ ]:
# How many liberals and conservatives are there total?
polviews_query = '''
SELECT PoliticalViews, COUNT(*) AS cnt
FROM `nyu-datasets.facebook.Profiles`
WHERE PoliticalViews IN ('Liberal', 'Conservative')
GROUP BY PoliticalViews
'''

df_polviews = bq.query(polviews_query).to_dataframe()
df_polviews = df_polviews.set_index('PoliticalViews')
df_polviews

In [ ]:
n_liberal = df_polviews.loc['Liberal', 'cnt']
n_conservative = df_polviews.loc['Conservative', 'cnt']

print(f"Liberals: {n_liberal:,}")
print(f"Conservatives: {n_conservative:,}")
print(f"Ratio: {n_conservative/n_liberal:.1f}x more conservatives")

### Calculate Percentages and Lift

In [ ]:
# Calculate percentage of each group that likes each book
# Add +1 to avoid division by zero (Laplace smoothing)
books_pivot['Liberal_pct'] = (books_pivot['Liberal'] + 1) / n_liberal * 100
books_pivot['Conservative_pct'] = (books_pivot['Conservative'] + 1) / n_conservative * 100

# Calculate lift
books_pivot['Lift_Liberal'] = books_pivot['Liberal_pct'] / books_pivot['Conservative_pct']
books_pivot['Lift_Conservative'] = books_pivot['Conservative_pct'] / books_pivot['Liberal_pct']

books_pivot.head(10)

### Interpreting Lift

A book with `Lift_Liberal = 2.0` means:
> "Liberals are **2x more likely** to list this as a favorite book compared to conservatives"

In [ ]:
# Most "liberal" books
liberal_books = (books_pivot
    .sort_values('Lift_Liberal', ascending=False)
    [['Liberal', 'Conservative', 'Lift_Liberal', 'Lift_Conservative']]
    .head(15)
)

print("=" * 60)
print("BOOKS PREFERRED BY LIBERALS")
print("(Higher Lift_Liberal = stronger liberal preference)")
print("=" * 60)
liberal_books

In [ ]:
# Most "conservative" books
conservative_books = (books_pivot
    .sort_values('Lift_Conservative', ascending=False)
    [['Liberal', 'Conservative', 'Lift_Liberal', 'Lift_Conservative']]
    .head(15)
)

print("=" * 60)
print("BOOKS PREFERRED BY CONSERVATIVES")
print("(Higher Lift_Conservative = stronger conservative preference)")
print("=" * 60)
conservative_books

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

liberal_books['Lift_Liberal'].sort_values().plot(
    kind='barh', ax=axes[0], color='blue', alpha=0.7
)
axes[0].set_title('Books Preferred by Liberals')
axes[0].set_xlabel('Lift (Liberal)')
axes[0].axvline(x=1, color='gray', linestyle='--', alpha=0.5)

conservative_books['Lift_Conservative'].sort_values().plot(
    kind='barh', ax=axes[1], color='red', alpha=0.7
)
axes[1].set_title('Books Preferred by Conservatives')
axes[1].set_xlabel('Lift (Conservative)')
axes[1].axvline(x=1, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()

---

## Part 4: Log-Odds — A Better Scale

Lift has an asymmetry problem:
- "2x more likely" (lift = 2) seems equivalent to "2x less likely" (lift = 0.5)
- But 2 ≠ -0.5

**Log-odds** fixes this by taking the logarithm:

$$\text{Log-Odds} = \log(\text{Lift})$$

| Lift | Log-Odds | Interpretation |
|------|----------|----------------|
| 2.0 | +0.30 | 2x more likely |
| 1.0 | 0.00 | No association |
| 0.5 | -0.30 | 2x less likely |

### Why Log-Odds?

1. **Symmetric**: +0.3 and -0.3 are equally strong associations
2. **Additive**: You can sum log-odds across multiple features
3. **Statistical**: Foundation for logistic regression

In [ ]:
# Calculate log-odds
books_pivot['LogOdds_Liberal'] = np.log10(books_pivot['Lift_Liberal'])
books_pivot['LogOdds_Conservative'] = np.log10(books_pivot['Lift_Conservative'])

# Note: LogOdds_Liberal = -LogOdds_Conservative (they're symmetric)
books_pivot[['Lift_Liberal', 'LogOdds_Liberal', 'Lift_Conservative', 'LogOdds_Conservative']].head(10)

In [ ]:
# Visualize log-odds distribution
plt.figure(figsize=(12, 5))
books_pivot['LogOdds_Liberal'].hist(bins=50, alpha=0.7, edgecolor='black')
plt.axvline(x=0, color='red', linestyle='--', linewidth=2, label='No association')
plt.xlabel('Log-Odds (Liberal)')
plt.ylabel('Number of Books')
plt.title('Distribution of Log-Odds\n(Negative = Conservative leaning, Positive = Liberal leaning)')
plt.legend()
plt.tight_layout()

---

## Part 5: Building a Scoring Model

The **additive property** of log-odds lets us build simple scoring models.

**Idea**: Sum the log-odds of a user's favorite books to estimate their political leaning.

$$\text{Political Score} = \sum_{\text{book} \in \text{favorites}} \text{LogOdds}_{\text{book}}$$

- Positive score → More likely liberal
- Negative score → More likely conservative

In [ ]:
# Create a lookup table of book scores
book_scores = books_pivot['LogOdds_Liberal'].to_dict()

# Show some examples
print("Sample book scores (positive = liberal, negative = conservative):")
for book in ['The Bible', 'Harry Potter', '1984', 'Catcher in the Rye']:
    if book in book_scores:
        print(f"  {book}: {book_scores[book]:.3f}")

In [ ]:
# Get users with their favorite books and known political views
users_query = '''
SELECT P.ProfileID, P.PoliticalViews, B.Book
FROM `nyu-datasets.facebook.Profiles` P
JOIN `nyu-datasets.facebook.FavoriteBooks` B ON B.ProfileID = P.ProfileID
WHERE P.PoliticalViews IN ('Very Liberal', 'Liberal', 'Moderate', 'Conservative', 'Very Conservative')
  AND B.Book IS NOT NULL
'''

df_users = bq.query(users_query).to_dataframe()
print(f"Loaded {len(df_users):,} user-book pairs")
df_users.head(10)

In [ ]:
# Add book scores
df_users['BookScore'] = df_users['Book'].map(book_scores).fillna(0)

# Calculate total score per user
user_scores = (df_users
    .groupby(['ProfileID', 'PoliticalViews'])
    .agg(
        TotalScore=('BookScore', 'sum'),
        NumBooks=('Book', 'count')
    )
    .reset_index()
)

user_scores.head(10)

In [ ]:
# How well does our score predict political views?
avg_scores = user_scores.groupby('PoliticalViews')['TotalScore'].mean()

# Order by political spectrum
order = ['Very Liberal', 'Liberal', 'Moderate', 'Conservative', 'Very Conservative']
avg_scores = avg_scores.reindex(order)

print("Average Political Score by Self-Reported Political View:")
print("(Positive = Liberal direction, Negative = Conservative direction)\n")
print(avg_scores.round(3))

In [ ]:
# Visualize
colors = ['darkblue', 'blue', 'purple', 'red', 'darkred']
avg_scores.plot(kind='barh', color=colors, figsize=(10, 4))
plt.axvline(x=0, color='black', linestyle='-', linewidth=1)
plt.xlabel('Average Book-Based Political Score')
plt.title('Validating the Scoring Model\n(Score should increase from Conservative to Liberal)')
plt.tight_layout()

**The model works!** Users who self-identify as more liberal have higher scores (more positive), and conservatives have lower scores (more negative).

This same technique powers:
- Spam filters (word log-odds)
- Recommendation systems (item affinity)
- Sentiment analysis (word sentiment scores)

---

## 🎯 Practice Activities

### Activity 1: Relationship Status and Looking For

Analyze the association between `RelationshipStatus` and `LookingFor` in the Facebook database.

1. Load data joining the `Relationship` and `LookingFor` tables
2. Create a pivot table
3. Normalize to find: "What are people in different relationship statuses looking for?"
4. Calculate lift to find the strongest associations

In [ ]:
# YOUR CODE HERE


### Activity 2: Music and Politics

Repeat the book analysis but for **favorite music** (`FavoriteMusic` table).

1. Find music preferences for liberals vs. conservatives
2. Calculate lift and log-odds
3. What music most strongly predicts political leaning?

In [ ]:
# YOUR CODE HERE


### Activity 3: Predict Unknown Political Views

Using the book scoring model from Part 5:

1. Find users who have NOT declared a political view but have listed favorite books
2. Calculate their political score
3. How many users score as "likely liberal" (score > 0.5) vs "likely conservative" (score < -0.5)?

In [ ]:
# YOUR CODE HERE


---

## 📝 Solutions

In [ ]:
# =============================================================================
# SOLUTION: Activity 1 - Relationship Status and Looking For
# =============================================================================

query = '''
SELECT R.Status, L.LookingFor, COUNT(*) AS cnt
FROM `nyu-datasets.facebook.Relationship` R
JOIN `nyu-datasets.facebook.LookingFor` L ON R.ProfileID = L.ProfileID
GROUP BY R.Status, L.LookingFor
HAVING cnt > 10
'''

df_rel = bq.query(query).to_dataframe()

# Pivot
pivot_rel = df_rel.pivot_table(
    index='Status',
    columns='LookingFor',
    values='cnt',
    aggfunc='sum'
).fillna(0)

print("Raw counts:")
display(pivot_rel)

# Normalize by relationship status (rows sum to 1)
normalized = pivot_rel.div(pivot_rel.sum(axis=1), axis=0)

print("\nNormalized (what each status is looking for):")
display(normalized.round(3))

# Visualize
normalized.plot(kind='barh', stacked=True, figsize=(12, 5))
plt.title('What People Are Looking For by Relationship Status')
plt.xlabel('Proportion')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

In [ ]:
# =============================================================================
# SOLUTION: Activity 2 - Music and Politics
# =============================================================================

music_query = '''
SELECT M.Music, P.PoliticalViews, COUNT(*) AS cnt
FROM `nyu-datasets.facebook.Profiles` P
JOIN `nyu-datasets.facebook.FavoriteMusic` M ON M.ProfileID = P.ProfileID
WHERE PoliticalViews IN ('Liberal', 'Conservative')
  AND M.Music IS NOT NULL
  AND M.Music IN (
      SELECT Music
      FROM `nyu-datasets.facebook.FavoriteMusic` M
      JOIN `nyu-datasets.facebook.Profiles` P ON M.ProfileID = P.ProfileID
      WHERE P.PoliticalViews IN ('Liberal', 'Conservative')
      GROUP BY Music
      HAVING COUNT(DISTINCT P.ProfileID) > 10
  )
GROUP BY M.Music, P.PoliticalViews
'''

df_music = bq.query(music_query).to_dataframe()

# Pivot and calculate lift
music_pivot = df_music.pivot_table(
    index='Music',
    columns='PoliticalViews',
    values='cnt'
).fillna(0)

music_pivot['Liberal_pct'] = (music_pivot['Liberal'] + 1) / n_liberal * 100
music_pivot['Conservative_pct'] = (music_pivot['Conservative'] + 1) / n_conservative * 100
music_pivot['Lift_Liberal'] = music_pivot['Liberal_pct'] / music_pivot['Conservative_pct']
music_pivot['LogOdds'] = np.log10(music_pivot['Lift_Liberal'])

# Top liberal music
print("Music preferred by LIBERALS:")
display(music_pivot.sort_values('Lift_Liberal', ascending=False)
        [['Liberal', 'Conservative', 'Lift_Liberal']].head(10))

# Top conservative music
print("\nMusic preferred by CONSERVATIVES:")
display(music_pivot.sort_values('Lift_Liberal', ascending=True)
        [['Liberal', 'Conservative', 'Lift_Liberal']].head(10))

In [ ]:
# =============================================================================
# SOLUTION: Activity 3 - Predict Unknown Political Views
# =============================================================================

# Get users with unknown political views
unknown_query = '''
SELECT P.ProfileID, B.Book
FROM `nyu-datasets.facebook.Profiles` P
JOIN `nyu-datasets.facebook.FavoriteBooks` B ON B.ProfileID = P.ProfileID
WHERE P.PoliticalViews IS NULL
  AND B.Book IS NOT NULL
'''

df_unknown = bq.query(unknown_query).to_dataframe()
print(f"Found {df_unknown['ProfileID'].nunique():,} users with unknown political views")

# Score them
df_unknown['BookScore'] = df_unknown['Book'].map(book_scores).fillna(0)

unknown_scores = (df_unknown
    .groupby('ProfileID')
    .agg(
        TotalScore=('BookScore', 'sum'),
        NumBooks=('Book', 'count')
    )
    .reset_index()
)

# Classify based on score
def classify(score):
    if score > 0.5:
        return 'Likely Liberal'
    elif score < -0.5:
        return 'Likely Conservative'
    else:
        return 'Uncertain'

unknown_scores['Prediction'] = unknown_scores['TotalScore'].apply(classify)

print("\nPredicted Political Leanings:")
print(unknown_scores['Prediction'].value_counts())

# Visualize distribution
plt.figure(figsize=(10, 4))
unknown_scores['TotalScore'].hist(bins=50, alpha=0.7, edgecolor='black')
plt.axvline(x=0.5, color='blue', linestyle='--', label='Liberal threshold')
plt.axvline(x=-0.5, color='red', linestyle='--', label='Conservative threshold')
plt.xlabel('Political Score')
plt.ylabel('Number of Users')
plt.title('Distribution of Predicted Political Scores\n(For Users Who Did Not Declare Political Views)')
plt.legend()
plt.tight_layout()

---

## Summary: Key Concepts

### Normalization

```python
# Column normalization (columns sum to 1)
normalized = df / df.sum()

# Row normalization (rows sum to 1)
normalized = df.div(df.sum(axis=1), axis=0)
# or
normalized = (df.T / df.T.sum()).T
```

### Lift

$$\text{Lift}(A, B) = \frac{P(A | B)}{P(A)} = \frac{\text{% of B that are A}}{\text{% of everyone that is A}}$$

```python
lift = percentage_in_group / percentage_overall
```

- Lift = 1: No association
- Lift > 1: Positive association
- Lift < 1: Negative association

### Log-Odds

$$\text{Log-Odds} = \log(\text{Lift})$$

```python
log_odds = np.log10(lift)  # or np.log(lift) for natural log
```

- Log-Odds = 0: No association
- Log-Odds > 0: Positive association
- Log-Odds < 0: Negative association
- **Additive**: Sum log-odds across features

### Applications

- **Market basket analysis**: "People who bought X also bought Y"
- **Recommendation systems**: Calculate item affinities
- **Text classification**: Word association with categories
- **A/B testing**: Segment performance analysis
- **Feature engineering**: Create association-based features